# 02 — Modelling with XGBoost

Trains a baseline XGBoost classifier on the processed (imbalanced) training data, then performs a brief Optuna hyperparameter search.

**Input:** `data/train_processed.csv`, `data/test_processed.csv`

**Output:** `models/xgb_baseline.json`, `models/xgb_tuned.json`, `models/best_params.json`

**Package versions assumed:** xgboost==1.7.6, scikit-learn==1.3.2, optuna (any recent version)

In [1]:
import json
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

RANDOM_SEED = 42
TARGET_COL = "at_risk"
os.makedirs("models", exist_ok=True)

c:\Users\Nahnah\OneDrive\Documents\Project_Work\dummy_smoke_test_package\pipeline_test\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load processed data

In [2]:
train_df = pd.read_csv("data/train_processed.csv")
test_df = pd.read_csv("data/test_processed.csv")

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]
X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (280, 35) | X_test: (70, 35)


## 2. Baseline XGBoost model

A baseline model trained with default-ish hyperparameters on the (imbalanced) training set. This establishes a reference point before SMOTE balancing (notebook 03) and hyperparameter tuning.

In [3]:
baseline_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=RANDOM_SEED,
)

baseline_model.fit(X_train, y_train)
print("Baseline model trained.")

Baseline model trained.


In [4]:
def evaluate_model(model, X, y, label="model"):
    preds = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    metrics = {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
    }
    print(f"--- {label} ---")
    for k, v in metrics.items():
        print(f"{k:>10}: {v:.4f}")
    print("\nConfusion matrix:")
    print(confusion_matrix(y, preds))
    print("\nClassification report:")
    print(classification_report(y, preds, zero_division=0))
    return metrics

baseline_metrics = evaluate_model(baseline_model, X_test, y_test, "Baseline XGBoost (test set)")

--- Baseline XGBoost (test set) ---
  accuracy: 0.7714
 precision: 0.7222
    recall: 0.5417
        f1: 0.6190
   roc_auc: 0.8723

Confusion matrix:
[[41  5]
 [11 13]]

Classification report:
              precision    recall  f1-score   support

           0       0.79      0.89      0.84        46
           1       0.72      0.54      0.62        24

    accuracy                           0.77        70
   macro avg       0.76      0.72      0.73        70
weighted avg       0.77      0.77      0.76        70



## 3. Hyperparameter tuning with Optuna

For the dummy-data smoke test, a small number of trials (`N_TRIALS`) is sufficient to confirm the search loop runs end-to-end. Increase `N_TRIALS` (e.g., to 50-100) once real data is available.

In [5]:
N_TRIALS = 15  # increase for real-data runs

def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": RANDOM_SEED,
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, proba)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

print("Best ROC-AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-06-14 00:33:17,676] A new study created in memory with name: no-name-7361d80a-6e18-4be0-b731-24b33b0a9519
[I 2026-06-14 00:33:18,234] Trial 0 finished with value: 0.875 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'gamma': 0.2904180608409973}. Best is trial 0 with value: 0.875.
[I 2026-06-14 00:33:19,333] Trial 1 finished with value: 0.845108695652174 and parameters: {'n_estimators': 267, 'max_depth': 6, 'learning_rate': 0.11114989443094977, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'min_child_weight': 9, 'gamma': 1.0616955533913808}. Best is trial 0 with value: 0.875.
[I 2026-06-14 00:33:19,616] Trial 2 finished with value: 0.8849637681159421 and parameters: {'n_estimators': 95, 'max_depth': 3, 'learning_rate': 0.028145092716060652, 'subsample': 0.8099025726528951, 'colsample_bytree': 0.7727780074568463, 'mi

Best ROC-AUC: 0.8894927536231884
Best params: {'n_estimators': 72, 'max_depth': 3, 'learning_rate': 0.011662890273931383, 'subsample': 0.7301321323053057, 'colsample_bytree': 0.7554709158757928, 'min_child_weight': 3, 'gamma': 4.143687545759647}


**Note:** For a small dataset, using the held-out test set inside the Optuna objective risks overfitting hyperparameters to the test set. For the real-data run, replace this with cross-validated AUC on the training set only (e.g., `StratifiedKFold` + `cross_val_score`), and reserve the test set purely for final evaluation in notebook 05.

## 4. Train tuned model and evaluate

In [6]:
best_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": RANDOM_SEED,
    **study.best_params,
}

tuned_model = xgb.XGBClassifier(**best_params)
tuned_model.fit(X_train, y_train)

tuned_metrics = evaluate_model(tuned_model, X_test, y_test, "Tuned XGBoost (test set)")

--- Tuned XGBoost (test set) ---
  accuracy: 0.7857
 precision: 0.7368
    recall: 0.5833
        f1: 0.6512
   roc_auc: 0.8895

Confusion matrix:
[[41  5]
 [10 14]]

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.89      0.85        46
           1       0.74      0.58      0.65        24

    accuracy                           0.79        70
   macro avg       0.77      0.74      0.75        70
weighted avg       0.78      0.79      0.78        70



## 5. Save models and best parameters

In [7]:
baseline_model.save_model("models/xgb_baseline.json")
tuned_model.save_model("models/xgb_tuned.json")

with open("models/best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2)

print("Saved baseline model, tuned model, and best params to ./models/")

Saved baseline model, tuned model, and best params to ./models/


## Smoke test checklist
- [ ] Baseline model trains and evaluates without errors
- [ ] Optuna study completes `N_TRIALS` trials
- [ ] Tuned model trains and evaluates without errors
- [ ] `models/xgb_baseline.json`, `models/xgb_tuned.json`, `models/best_params.json` are created

Proceed to **03_smote_imbalance.ipynb**.